In [1]:
import pandas as pd
from datetime import datetime

fighter_stats = pd.read_csv("ufc-fighters-statistics.csv")
fighter_stats.head()

# Drop column 'nickname'
fighter_stats = fighter_stats.drop(columns=['nickname'])

# Replace missing values with the median in 'height_cm'
fighter_stats = fighter_stats.fillna({'height_cm': fighter_stats['height_cm'].median()})

# Replace missing values with the median in 'weight_in_kg'
fighter_stats = fighter_stats.fillna({'weight_in_kg': fighter_stats['weight_in_kg'].median()})

# Replace missing values with the mean, as it is balanced, in 'weight_in_kg'
fighter_stats = fighter_stats.fillna({'weight_in_kg': fighter_stats['weight_in_kg'].mean()})

# Replace missing values with the mean, as it is balanced, in 'reach_in_cm'
fighter_stats = fighter_stats.fillna({'reach_in_cm': fighter_stats['reach_in_cm'].mean()})

# Change 'date_of_birth' column to 'age'
fighter_stats['date_of_birth'] = pd.to_datetime(fighter_stats['date_of_birth'])

today = pd.Timestamp.today()

def calculate_age(dob):
    if pd.isnull(dob):
        return 0
    return today.year - dob.year - ((today.month, today.day) < (dob.month, dob.day))
fighter_stats['age'] = fighter_stats['date_of_birth'].apply(calculate_age)

fighter_stats = fighter_stats.drop(columns=['date_of_birth']) # Information changed to numeric 'age' column so can drop 'date_of_birth' column

# Replace missing values with the median in 'age'
fighter_stats = fighter_stats.fillna({'age': fighter_stats['age'].median()})

# Replace missing values with the median in 'significant_strikes_landed_per_minute'
fighter_stats = fighter_stats.fillna({'significant_strikes_landed_per_minute': fighter_stats['significant_strikes_landed_per_minute'].median()})

# Replace missing values with the median in 'significant_striking_accuracy'
fighter_stats = fighter_stats.fillna({'significant_striking_accuracy': fighter_stats['significant_striking_accuracy'].median()})

# Replace missing values with the median in 'significant_strikes_absorbed'
fighter_stats = fighter_stats.fillna({'significant_strikes_absorbed_per_minute': fighter_stats['significant_strikes_absorbed_per_minute'].median()})

# Replace missing values with the median in 'significant_strike_defence'
fighter_stats = fighter_stats.fillna({'significant_strike_defence': fighter_stats['significant_strike_defence'].median()})

# Replace missing values with the median in 'average_takedowns_landed_per_15_minutes'
fighter_stats = fighter_stats.fillna({'average_takedowns_landed_per_15_minutes': fighter_stats['average_takedowns_landed_per_15_minutes'].median()})

# Replace missing values with the median in 'takedown_accuracy'
fighter_stats = fighter_stats.fillna({'takedown_accuracy': fighter_stats['takedown_accuracy'].median()})

# Replace missing values with the median in 'takedown_defense'
fighter_stats = fighter_stats.fillna({'takedown_defense': fighter_stats['takedown_defense'].median()})

# Replace missing values with the median in 'average_submissions_attempted_per_15_minutes'
fighter_stats = fighter_stats.fillna({'average_submissions_attempted_per_15_minutes': fighter_stats['average_submissions_attempted_per_15_minutes'].median()})

# Replace missing values with 'Unknown' in 'stance'
fighter_stats = fighter_stats.fillna({'stance': 'Unknown'})

# One-Hot Encoding to encode the 'stance' column
fighter_stats = pd.get_dummies(fighter_stats, columns=['stance'], prefix='stance', dtype=int)
fighter_stats


,name,wins,losses,draws,height_cm,weight_in_kg,reach_in_cm,significant_strikes_landed_per_minute,significant_striking_accuracy,significant_strikes_absorbed_per_minute,...,takedown_accuracy,takedown_defense,average_submissions_attempted_per_15_minutes,age,stance_Open Stance,stance_Orthodox,stance_Sideways,stance_Southpaw,stance_Switch,stance_Unknown
0,Robert Drysdale,7,0,0,190.50,92.99,181.808874,0.00,0.0,0.00,...,100.0,0.0,21.9,43,0,1,0,0,0,0
1,Daniel McWilliams,15,37,0,185.42,83.91,181.808874,3.36,77.0,0.00,...,0.0,100.0,21.6,0,0,0,0,0,0,1
2,Dan Molina,13,9,0,177.80,97.98,181.808874,0.00,0.0,5.58,...,0.0,0.0,20.9,0,0,0,0,0,0,1
3,Paul Ruiz,7,4,0,167.64,61.23,181.808874,1.40,33.0,1.40,...,0.0,100.0,20.9,0,0,0,0,0,0,1
4,Collin Huckbody,8,2,0,190.50,83.91,193.040000,2.05,60.0,2.73,...,100.0,0.0,20.4,30,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4106,John Campetella,0,1,0,175.26,106.59,181.808874,0.00,0.0,0.00,...,0.0,0.0,0.0,0,0,1,0,0,0,0
4107,Andre Pederneiras,1,1,2,172.72,70.31,181.808874,0.00,0.0,0.00,...,0.0,0.0,0.0,58,0,1,0,0,0,0
4108,Bryson Kamaka,12,20,1,180.34,77.11,181.808874,9.47,60.0,12.63,...,0.0,100.0,0.0,0,0,1,0,0,0,0
4109,Matej Penaz,6,1,0,190.50,83.91,210.820000,1.28,33.0,2.55,...,0.0,0.0,0.0,28,0,0,0,1,0,0


In [2]:
def clean_name(name):
    return name.lower().strip().replace('.', '').replace("'", "")

fighter_stats['name'] = fighter_stats['name'].apply(clean_name)

fight_results = pd.read_csv("ufc_fight_results.csv")

fight_results[['fighter_1', 'fighter_2']] = fight_results['BOUT'].str.split('vs. ', expand=True)
fight_results['fighter_1'] = fight_results['fighter_1'].apply(clean_name)
fight_results['fighter_2'] = fight_results['fighter_2'].apply(clean_name)

# Melt both sides of the fight so each row represents a fighter
fight_df1 = fight_results.rename(columns={'fighter_1': 'name', 'OUTCOME': 'outcome_raw'})
fight_df2 = fight_results.rename(columns={'fighter_2': 'name', 'OUTCOME': 'outcome_raw'})

fight_df1['outcome'] = fight_df1['outcome_raw'].str[0]  # 'W' or 'L'
fight_df2['outcome'] = fight_df2['outcome_raw'].str[-1] # 'L' or 'W'

all_fights = pd.concat([fight_df1, fight_df2])[['EVENT', 'name', 'outcome', 'WEIGHTCLASS', 'METHOD', 'ROUND', 'TIME']]
all_fights = all_fights.sort_values(by='EVENT') # Switch to ordinal encoding when possible

fight_results


,EVENT,BOUT,OUTCOME,WEIGHTCLASS,METHOD,ROUND,TIME,TIME FORMAT,REFEREE,DETAILS,URL,fighter_1,fighter_2
0,UFC 314: Volkanovski vs. Lopes,Alexander Volkanovski vs. Diego Lopes,W/L,UFC Featherweight Title Bout,Decision - Unanimous,5,5:00,5 Rnd (5-5-5-5-5),Marc Goddard,Sal D'amato 46 - 49.Chris Lee 46 - 49.Derek Cl...,http://ufcstats.com/fight-details/e733f148060b...,alexander volkanovski,diego lopes
1,UFC 314: Volkanovski vs. Lopes,Michael Chandler vs. Paddy Pimblett,L/W,Lightweight Bout,KO/TKO,3,3:07,5 Rnd (5-5-5-5-5),Kerry Hatley,Elbows to Head From Mount,http://ufcstats.com/fight-details/d05cb4c4135c...,michael chandler,paddy pimblett
2,UFC 314: Volkanovski vs. Lopes,Yair Rodriguez vs. Patricio Freire,W/L,Featherweight Bout,Decision - Unanimous,3,5:00,3 Rnd (5-5-5),Andrew Glenn,Eliseo Rodriguez 27 - 30.Junichiro Kamijo 27 -...,http://ufcstats.com/fight-details/d3be5a4e0ec2...,yair rodriguez,patricio freire
3,UFC 314: Volkanovski vs. Lopes,Bryce Mitchell vs. Jean Silva,L/W,Featherweight Bout,Submission,2,3:52,3 Rnd (5-5-5),Mike Beltran,Guillotine Choke In Clinch,http://ufcstats.com/fight-details/8c540eb4afe8...,bryce mitchell,jean silva
4,UFC 314: Volkanovski vs. Lopes,Nikita Krylov vs. Dominick Reyes,L/W,Light Heavyweight Bout,KO/TKO,1,2:24,3 Rnd (5-5-5),Marc Goddard,Punch to Head At Distance,http://ufcstats.com/fight-details/b2d731415bd3...,nikita krylov,dominick reyes
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8076,UFC 2: No Way Out,Orlando Wiet vs. Robert Lucarelli,W/L,Open Weight Bout,KO/TKO,1,2:50,No Time Limit,John McCarthy,toCorner Stoppage,http://ufcstats.com/fight-details/3b020d4914b4...,orlando wiet,robert lucarelli
8077,UFC 2: No Way Out,Frank Hamaker vs. Thaddeus Luster,W/L,Open Weight Bout,Submission,1,4:52,No Time Limit,John McCarthy,Keylock From Half Guard,http://ufcstats.com/fight-details/d917c8c7461b...,frank hamaker,thaddeus luster
8078,UFC 2: No Way Out,Johnny Rhodes vs. David Levicki,W/L,Open Weight Bout,KO/TKO,1,12:13,No Time Limit,John McCarthy,Punches to Head From GuardSubmission to Strikes,http://ufcstats.com/fight-details/ccee020be2e8...,johnny rhodes,david levicki
8079,UFC 2: No Way Out,Patrick Smith vs. Ray Wizard,W/L,Open Weight Bout,Submission,1,0:58,No Time Limit,John McCarthy,Guillotine Choke Standing,http://ufcstats.com/fight-details/4b9ae533ccb3...,patrick smith,ray wizard


In [3]:
# Rolling win rate
all_fights['win'] = all_fights['outcome'] == 'W'
all_fights['win'] = all_fights['win'].astype(int)


all_fights['rolling_win_rate'] = (
    all_fights
    .groupby('name')['win']
    .transform(lambda x: x.rolling(5, min_periods=1).mean().shift())
)

all_fights

,EVENT,name,outcome,WEIGHTCLASS,METHOD,ROUND,TIME,win,rolling_win_rate
7450,Ortiz vs Shamrock 3: The Final Chapter,nate marquardt,W,Middleweight Bout,Submission,2,1:14,1,NaN
7448,Ortiz vs Shamrock 3: The Final Chapter,jason macdonald,W,Middleweight Bout,Submission,1,2:43,1,NaN
7451,Ortiz vs Shamrock 3: The Final Chapter,tony desouza,W,Welterweight Bout,Submission,1,3:59,1,NaN
7454,Ortiz vs Shamrock 3: The Final Chapter,forrest petz,L,Welterweight Bout,Submission,1,4:58,0,NaN
7453,Ortiz vs Shamrock 3: The Final Chapter,john alessio,L,Welterweight Bout,Decision - Unanimous,3,5:00,0,NaN
...,...,...,...,...,...,...,...,...,...
7122,UFC: Silva vs Irvin,hermes franca,L,Lightweight Bout,Decision - Unanimous,3,5:00,0,0.8
7120,UFC: Silva vs Irvin,anderson silva,W,Light Heavyweight Bout,KO/TKO,1,1:01,1,0.6
7130,UFC: Silva vs Irvin,dale hartt,L,Lightweight Bout,Submission,1,3:33,0,0.5
7123,UFC: Silva vs Irvin,jake obrien,L,Heavyweight Bout,KO/TKO,1,2:02,0,0.8


In [4]:
# Drop NA in case some fighters only had 1 match, etc.
latest_rolling_win = (
    all_fights
    .dropna(subset=['rolling_win_rate'])
    .groupby('name', as_index=False)
    .tail(1)  # gets the last fight entry (i.e., most recent rolling stat)
    [['name', 'rolling_win_rate']]
    .drop_duplicates(subset='name')  # ensure no duplicates just in case
)

fighter_stats = fighter_stats.merge(
    latest_rolling_win,
    on='name',
    how='left'
)


fighter_stats

,name,wins,losses,draws,height_cm,weight_in_kg,reach_in_cm,significant_strikes_landed_per_minute,significant_striking_accuracy,significant_strikes_absorbed_per_minute,...,takedown_defense,average_submissions_attempted_per_15_minutes,age,stance_Open Stance,stance_Orthodox,stance_Sideways,stance_Southpaw,stance_Switch,stance_Unknown,rolling_win_rate
0,robert drysdale,7,0,0,190.50,92.99,181.808874,0.00,0.0,0.00,...,0.0,21.9,43,0,1,0,0,0,0,NaN
1,daniel mcwilliams,15,37,0,185.42,83.91,181.808874,3.36,77.0,0.00,...,100.0,21.6,0,0,0,0,0,0,1,NaN
2,dan molina,13,9,0,177.80,97.98,181.808874,0.00,0.0,5.58,...,0.0,20.9,0,0,0,0,0,0,1,NaN
3,paul ruiz,7,4,0,167.64,61.23,181.808874,1.40,33.0,1.40,...,100.0,20.9,0,0,0,0,0,0,1,NaN
4,collin huckbody,8,2,0,190.50,83.91,193.040000,2.05,60.0,2.73,...,0.0,20.4,30,0,1,0,0,0,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4106,john campetella,0,1,0,175.26,106.59,181.808874,0.00,0.0,0.00,...,0.0,0.0,0,0,1,0,0,0,0,NaN
4107,andre pederneiras,1,1,2,172.72,70.31,181.808874,0.00,0.0,0.00,...,0.0,0.0,58,0,1,0,0,0,0,NaN
4108,bryson kamaka,12,20,1,180.34,77.11,181.808874,9.47,60.0,12.63,...,100.0,0.0,0,0,1,0,0,0,0,NaN
4109,matej penaz,6,1,0,190.50,83.91,210.820000,1.28,33.0,2.55,...,0.0,0.0,28,0,0,0,1,0,0,NaN


In [5]:
fighter_stats['rolling_win_rate'] = fighter_stats['rolling_win_rate'].fillna(fighter_stats['rolling_win_rate'].mean())

fighter_stats[fighter_stats['name'] == 'diego lopes']
fight_data = None

In [6]:
fight_data = fight_results.merge(
    fighter_stats, left_on="fighter_1", right_on='name', how='left'
).add_prefix('1_')

fight_data = fight_data.merge(
    fighter_stats, left_on='1_fighter_2', right_on='name', how='left'
).add_prefix('2_')

fight_data = fight_data.drop(columns=['2_1_BOUT', '2_1_WEIGHTCLASS', '2_1_METHOD', '2_1_BOUT', '2_1_ROUND', '2_1_TIME', '2_1_TIME FORMAT', '2_1_REFEREE', '2_1_DETAILS', '2_1_URL', '2_1_fighter_1', '2_1_fighter_2'])

In [7]:
fight_data['outcome'] = fight_data['2_1_OUTCOME'].str[0]  # 'W' or 'L'

fight_data['outcome'] = (fight_data['outcome'] == 'W').astype(int)

fight_data = fight_data.drop(columns=['2_1_OUTCOME'])
fight_data

,2_1_EVENT,2_1_name,2_1_wins,2_1_losses,2_1_draws,2_1_height_cm,2_1_weight_in_kg,2_1_reach_in_cm,2_1_significant_strikes_landed_per_minute,2_1_significant_striking_accuracy,...,2_average_submissions_attempted_per_15_minutes,2_age,2_stance_Open Stance,2_stance_Orthodox,2_stance_Sideways,2_stance_Southpaw,2_stance_Switch,2_stance_Unknown,2_rolling_win_rate,outcome
0,UFC 314: Volkanovski vs. Lopes,alexander volkanovski,26.0,3.0,0.0,167.64,65.77,180.340000,6.19,57.0,...,5.3,30.0,0.0,1.0,0.0,0.0,0.0,0.0,0.800000,1
1,UFC 314: Volkanovski vs. Lopes,michael chandler,23.0,8.0,0.0,172.72,70.31,180.340000,4.89,46.0,...,2.4,30.0,0.0,1.0,0.0,0.0,0.0,0.0,1.000000,0
2,UFC 314: Volkanovski vs. Lopes,yair rodriguez,16.0,4.0,0.0,180.34,65.77,180.340000,4.63,46.0,...,0.6,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.409243,1
3,UFC 314: Volkanovski vs. Lopes,bryce mitchell,16.0,2.0,0.0,177.80,65.77,177.800000,2.34,58.0,...,0.0,28.0,0.0,1.0,0.0,0.0,0.0,0.0,1.000000,0
4,UFC 314: Volkanovski vs. Lopes,bryce mitchell,16.0,2.0,0.0,177.80,65.77,177.800000,2.34,58.0,...,0.0,47.0,0.0,1.0,0.0,0.0,0.0,0.0,1.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8113,UFC 2: No Way Out,orlando wiet,1.0,5.0,0.0,177.80,77.11,181.808874,0.00,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.409243,1
8114,UFC 2: No Way Out,frank hamaker,1.0,0.0,0.0,177.80,77.11,181.808874,0.00,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.409243,1
8115,UFC 2: No Way Out,johnny rhodes,2.0,1.0,0.0,182.88,95.25,181.808874,0.00,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.409243,1
8116,UFC 2: No Way Out,patrick smith,20.0,17.0,0.0,187.96,102.06,181.808874,0.00,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.409243,1


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import pandas as pd

fight_data = pd.read_csv("fight_data_final.csv")

X = fight_data.drop(columns=['outcome', '2_1_name', '2_name', '2_1_EVENT'])
X = X.fillna(X.mean(numeric_only=True))
y = fight_data['outcome']


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

clf = LogisticRegression()
clf.fit(X_train_scaled, y_train)

y_pred = clf.predict(X_test_scaled)
y_proba = clf.predict_proba(X_test_scaled)[:, 1]

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


print("\nROC AUC Score:", roc_auc_score(y_test, y_proba))

Confusion Matrix:
[[343 249]
 [173 859]]

Classification Report:
              precision    recall  f1-score   support

           0       0.66      0.58      0.62       592
           1       0.78      0.83      0.80      1032

    accuracy                           0.74      1624
   macro avg       0.72      0.71      0.71      1624
weighted avg       0.73      0.74      0.74      1624


ROC AUC Score: 0.8036767690655773


In [9]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=6)
knn.fit(X_train_scaled, y_train)

y_knn_pred = knn.predict(X_test_scaled)
y_knn_proba = knn.predict_proba(X_test_scaled)[:, 1]

print("K Nearest Confusion Matrix:\n", confusion_matrix(y_test, y_knn_pred))
print("\nK Nearest Classification Report:\n", classification_report(y_test, y_knn_pred))
print("\nROC AUC Score:", roc_auc_score(y_test, y_knn_proba))

K Nearest Confusion Matrix:
 [[373 219]
 [311 721]]

K Nearest Classification Report:
               precision    recall  f1-score   support

           0       0.55      0.63      0.58       592
           1       0.77      0.70      0.73      1032

    accuracy                           0.67      1624
   macro avg       0.66      0.66      0.66      1624
weighted avg       0.69      0.67      0.68      1624


ROC AUC Score: 0.709819557930023


In [10]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

y_rf_pred = rf.predict(X_test)
y_rf_proba = rf.predict_proba(X_test)[:, 1]  # for ROC AUC

print("Random Forest - Confusion Matrix:")
print(confusion_matrix(y_test, y_rf_pred))

print("\nRandom Forest - Classification Report:")
print(classification_report(y_test, y_rf_pred))

print("\nRandom Forest - ROC AUC Score:", roc_auc_score(y_test, y_rf_proba))

Random Forest - Confusion Matrix:
[[331 261]
 [164 868]]

Random Forest - Classification Report:
              precision    recall  f1-score   support

           0       0.67      0.56      0.61       592
           1       0.77      0.84      0.80      1032

    accuracy                           0.74      1624
   macro avg       0.72      0.70      0.71      1624
weighted avg       0.73      0.74      0.73      1624


Random Forest - ROC AUC Score: 0.8118346362350723


In [26]:
fight_data['outcome'].value_counts()
X.shape[0]

8118

In [44]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1) Prepare & scale
X_np = X.to_numpy(dtype=np.float32)
y_np = y.to_numpy(dtype=np.float32)
X_tr, X_te, y_tr, y_te = train_test_split(X_np, y_np, test_size=0.2,
                                         stratify=y_np, random_state=42)
scaler = StandardScaler().fit(X_tr)
X_tr = scaler.transform(X_tr)
X_te = scaler.transform(X_te)

# 2) Model
class MLP(nn.Module):
    def __init__(self, in_dim):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(in_dim, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 1)

    def forward(self, x):
        h0 = x.view(x.size(0), -1)
        h1 = F.relu(self.fc1(h0))
        h2 = F.relu(self.fc2(h1))
        h3 = self.fc3(h2)
        return h3

model = MLP(X_tr.shape[1])
model.cuda()

n_neg = (y_tr == 0).sum()
n_pos = (y_tr == 1).sum()
# how much more to weight a positive
pos_weight = torch.tensor([n_neg / n_pos], device=device, dtype=torch.float32)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
# optimizer
optimizer = torch.optim.Adam(model.parameters(), lr = 0.01)

from torch.utils.data import WeightedRandomSampler

# 1) Compute per-sample weights = inverse class frequency
#    y_tr is your NumPy array of 0/1 labels
class_counts   = np.bincount(y_tr.astype(int))        # e.g. [n_neg, n_pos]
class_weights  = 1. / class_counts                    # array([1/n_neg, 1/n_pos])
sample_weights = class_weights[y_tr.astype(int)]       # length n_samples

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

from torch.utils.data import TensorDataset, DataLoader

train_ds = TensorDataset(torch.from_numpy(X_tr).float().to(device), torch.from_numpy(y_tr).float().to(device))
test_ds = TensorDataset(torch.from_numpy(X_te).float().to(device), torch.from_numpy(y_te).float().to(device))
train_loader  = DataLoader(train_ds, batch_size=100, sampler=sampler, drop_last=True)
test_loader = DataLoader(test_ds, batch_size=100, shuffle=False)


def train(epoch):
  model.train()
  for batch_idx, (data, target) in enumerate(train_loader):
    data, target = data.cuda(), target.cuda()

    output = model(data)
    output = output.squeeze(1)
    loss = criterion(output, target)

    optimizer.zero_grad()
    loss.backward()

    optimizer.step()

    if batch_idx % 10 == 0:
      print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, batch_idx * len(data), len(train_loader.dataset),
                100. * batch_idx / len(train_loader), loss.item()))

def test():
    model.eval()
    test_loss = 0.0
    correct   = 0
    total     = 0

    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.cuda(), target.cuda().float()

            # 1) Forward pass
            logits = model(data).squeeze(1)  # shape (batch_size,)

            # 2) Loss
            loss = criterion(logits, target)
            test_loss += loss.item() * data.size(0)

            # 3) Compute predictions by thresholding at 0.5
            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).long()    # shape (batch_size,)

            # 4) Accumulate accuracy
            correct += preds.eq(target.long()).sum().item()
            total   += data.size(0)

    # 5) Averages
    test_loss /= total
    accuracy  = correct / total * 100

    print(
        f"\nTest set: Average loss: {test_loss:.4f}, "
        f"Accuracy: {correct}/{total} ({accuracy:.1f}%)\n"
    )

for epoch in range(1, 11):
  train(epoch)
  test()

# 5) Evaluate
model.eval()
with torch.no_grad():
    logits = model(torch.from_numpy(X_te).float().to(device))
    probs  = torch.sigmoid(logits).cpu().numpy()
auc = roc_auc_score(y_te, probs)
y_pred = (probs >- 0.5).astype(int)
y_true = y_te.astype(int)
print("Test AUC:", auc)
print("Classification Reports: \n", classification_report(y_true, y_pred))


Train Epoch: 1 [0/6494 (0%)]	Loss: 0.531912
Train Epoch: 1 [1000/6494 (16%)]	Loss: 0.459102
Train Epoch: 1 [2000/6494 (31%)]	Loss: 0.380401
Train Epoch: 1 [3000/6494 (47%)]	Loss: 0.351288
Train Epoch: 1 [4000/6494 (62%)]	Loss: 0.451573
Train Epoch: 1 [5000/6494 (78%)]	Loss: 0.407020
Train Epoch: 1 [6000/6494 (94%)]	Loss: 0.439458

Test set: Average loss: 0.4131, Accuracy: 1027/1624 (63.2%)

Train Epoch: 2 [0/6494 (0%)]	Loss: 0.379230
Train Epoch: 2 [1000/6494 (16%)]	Loss: 0.371650
Train Epoch: 2 [2000/6494 (31%)]	Loss: 0.406930
Train Epoch: 2 [3000/6494 (47%)]	Loss: 0.454379
Train Epoch: 2 [4000/6494 (62%)]	Loss: 0.351550
Train Epoch: 2 [5000/6494 (78%)]	Loss: 0.341762
Train Epoch: 2 [6000/6494 (94%)]	Loss: 0.394474

Test set: Average loss: 0.4374, Accuracy: 1085/1624 (66.8%)

Train Epoch: 3 [0/6494 (0%)]	Loss: 0.398642
Train Epoch: 3 [1000/6494 (16%)]	Loss: 0.396955
Train Epoch: 3 [2000/6494 (31%)]	Loss: 0.316848
Train Epoch: 3 [3000/6494 (47%)]	Loss: 0.333371
Train Epoch: 3 [4000/649

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

rf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)
print("RF AUC:", cross_val_score(rf, X, y, cv=5, scoring='roc_auc').mean())

RF AUC: 0.7574304140999136
